# 02 - Advanced Pandas & Data Wrangling
**Parts E + F:** MultiIndex, merge, join, concat, pivot, crosstab, apply/lambda, window operations, reshaping, encoding, data integration.
Every function lives in `scripts/wrangling.py`.

In [1]:
import sys
sys.path.insert(0, "../scripts")          # reuse the project modules
import pandas as pd, numpy as np
from sqlalchemy import text
import db
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_columns = 30
engine = db.get_engine()
import wrangling as W
from features import load_sales
df = load_sales()          # SELECT * FROM vw_sales_detail
print(df.shape); df.head(3)

(5905, 18)


,item_id,order_id,order_date,year,quarter,month,iso_weekday,year_month,customer_id,customer_name,region,category,product,payment_method,quantity,unit_price,discount,revenue
0,1,53642,2024-01-01,2024,1,1,1,2024-01,CUST-1193,Priya Muller,South,Office Supplies,Printer Paper,Debit Card,2,25.86,0.00,51.72
1,2,51997,2024-01-01,2024,1,1,1,2024-01,CUST-1102,Emma Reddy,West,Technology,Smartphone,Credit Card,1,530.24,0.05,503.73
2,3,51975,2024-01-01,2024,1,1,1,2024-01,CUST-1101,Emma Wang,Central,Clothing,Jacket,Credit Card,2,96.21,0.00,192.42


## 1. MultiIndex

In [2]:
mi = W.multiindex_summary(df)
mi.head(8)

revenue  orders  avg_discount
region  category                                        
Central Clothing         11,954.63      97          0.04
        Furniture        34,418.93      91          0.05
        Home & Kitchen   10,244.33      61          0.05
        Office Supplies   6,392.95     113          0.06
        Technology       74,708.81     134          0.05
East    Clothing         53,098.76     443          0.05
        Furniture       116,328.90     298          0.05
        Home & Kitchen   51,554.89     339          0.05

In [3]:
print("Select one region:"); print(mi.loc["East"].head(3))
print("\nCross-section: every region's Technology row:"); print(mi.xs("Technology", level="category"))
print("\nunstack() moves the inner index to columns:"); mi["revenue"].unstack().round(0)

Select one region:
                  revenue  orders  avg_discount
category                                       
Clothing        53,098.76     443          0.05
Furniture      116,328.90     298          0.05
Home & Kitchen  51,554.89     339          0.05

Cross-section: every region's Technology row:
           revenue  orders  avg_discount
region                                  
Central  74,708.81     134          0.05
East    299,991.45     537          0.05
North   136,393.58     259          0.05
South   152,230.60     274          0.05
West    148,305.37     288          0.06

unstack() moves the inner index to columns:


category,Clothing,Furniture,Home & Kitchen,Office Supplies,Technology
region,,,,,
Central,"11,955.00","34,419.00","10,244.00","6,393.00","74,709.00"
East,"53,099.00","116,329.00","51,555.00","31,647.00","299,991.00"
North,"23,790.00","49,776.00","23,254.00","17,339.00","136,394.00"
South,"29,943.00","68,632.00","24,554.00","16,762.00","152,231.00"
West,"27,029.00","71,466.00","23,884.00","16,563.00","148,305.00"


In [4]:
W.multiindex_pivot(df).head(12)

year                          2024       2025      TOTAL
region  category                                        
Central Clothing          5,418.00   6,537.00  11,955.00
        Furniture        16,657.00  17,762.00  34,419.00
        Home & Kitchen    6,470.00   3,774.00  10,244.00
        Office Supplies   2,632.00   3,761.00   6,393.00
        Technology       38,812.00  35,896.00  74,709.00
East    Clothing         26,826.00  26,273.00  53,099.00
        Furniture        56,138.00  60,190.00 116,329.00
        Home & Kitchen   27,630.00  23,925.00  51,555.00
        Office Supplies  15,605.00  16,042.00  31,647.00
        Technology      141,862.00 158,129.00 299,991.00
North   Clothing         12,056.00  11,734.00  23,790.00
        Furniture        22,104.00  27,672.00  49,776.00

## 2. Merge, join, concat

In [5]:
m = W.merge_customer_summary(df)       # MERGE (SQL-style join on a key)
m[["order_id", "customer_name", "revenue", "lifetime_revenue", "pct_of_customer_lifetime"]].head()

,order_id,customer_name,revenue,lifetime_revenue,pct_of_customer_lifetime
0,53642,Priya Muller,51.72,"21,032.52",0.25
1,51997,Emma Reddy,503.73,"2,266.39",22.23
2,51975,Emma Wang,192.42,"7,987.23",2.41
3,51505,Fatima Nair,81.91,"14,942.74",0.55
4,55544,Carlos Nair,40.32,"1,689.38",2.39


In [6]:
W.join_on_index(df)                    # JOIN on the index

,revenue,customers,revenue_per_customer
region,,,
East,"552,621.13",85,"6,501.00"
South,"292,122.32",45,"6,492.00"
West,"287,247.62",69,"4,163.00"
North,"250,553.26",59,"4,247.00"
Central,"137,719.65",34,"4,051.00"


In [7]:
W.concat_years(df)                     # CONCAT year columns side by side

,2024,2025
category,,
Clothing,"69,580.00","76,236.00"
Furniture,"167,860.00","172,763.00"
Home & Kitchen,"67,841.00","65,650.00"
Office Supplies,"42,916.00","45,788.00"
Technology,"394,951.00","416,679.00"


## 3. Crosstab, apply and lambda

In [8]:
W.region_payment_crosstab(df)          # % of each region's revenue by payment method

payment_method,Cash,Credit Card,Debit Card,Paypal
region,,,,
Central,15.60,34.70,32.80,16.80
East,13.50,40.90,31.40,14.20
North,14.90,38.00,31.50,15.60
South,15.00,41.10,28.10,15.70
West,13.70,36.70,34.70,14.80


In [9]:
W.apply_lambda_features(df).head()

,order_id,customer_name,product,quantity,revenue,order_size,initials,revenue_per_unit
0,53642,Priya Muller,Printer Paper,2,51.72,Small,PM,25.86
1,51997,Emma Reddy,Smartphone,1,503.73,Medium,ER,503.73
2,51975,Emma Wang,Jacket,2,192.42,Small,EW,96.21
3,51505,Fatima Nair,Sneakers,1,81.91,Small,FN,81.91
4,55544,Carlos Nair,Notebook Pack,4,40.32,Small,CN,10.08


## 4. Window operations (rolling, expanding, shift, pct_change, rank)

In [10]:
W.window_operations(df).tail(8)

,year_month,revenue,rolling_3m_avg,expanding_total,prev_month,mom_growth_pct,revenue_rank
16,2025-05,"55,191.00","57,602.00","1,016,844.00","57,937.00",-5.00,18
17,2025-06,"60,655.00","57,928.00","1,077,499.00","55,191.00",10.00,11
18,2025-07,"63,527.00","59,791.00","1,141,026.00","60,655.00",5.00,9
19,2025-08,"57,436.00","60,540.00","1,198,462.00","63,527.00",-10.00,16
20,2025-09,"57,838.00","59,601.00","1,256,301.00","57,436.00",1.00,15
21,2025-10,"74,337.00","63,204.00","1,330,637.00","57,838.00",28.00,4
22,2025-11,"95,930.00","76,035.00","1,426,567.00","74,337.00",29.00,2
23,2025-12,"93,697.00","87,988.00","1,520,264.00","95,930.00",-2.00,3


## 5. Reshaping: pivot / melt / stack

In [11]:
wide, long, stacked = W.reshape_wide_long(df)
print("WIDE (month x category):", wide.shape); display(wide.head(3))
print("LONG (tidy):", long.shape); display(long.head(3))
print("STACKED Series with MultiIndex:"); stacked.head(4)

WIDE (month x category): (24, 5)


category,Clothing,Furniture,Home & Kitchen,Office Supplies,Technology
year_month,,,,,
2024-01,"4,602.00","6,258.00","5,828.00","1,811.00","16,249.00"
2024-02,"3,955.00","11,892.00","4,788.00","2,573.00","26,523.00"
2024-03,"3,727.00","18,961.00","4,783.00","2,536.00","23,092.00"


LONG (tidy): (120, 3)


,year_month,category,revenue
0,2024-01,Clothing,"4,602.00"
1,2024-02,Clothing,"3,955.00"
2,2024-03,Clothing,"3,727.00"


STACKED Series with MultiIndex:


year_month  category       
2024-01     Clothing          4,602.00
            Furniture         6,258.00
            Home & Kitchen    5,828.00
            Office Supplies   1,811.00
dtype: float64

## 6. Encoding categorical data

In [12]:
W.encode_categories(df).head()

,cat_Clothing,cat_Furniture,cat_Home & Kitchen,cat_Office Supplies,cat_Technology,reg_Central,reg_East,reg_North,reg_South,reg_West,price_tier_ordinal
0,0,0,0,1,0,0,0,0,1,0,1
1,0,0,0,0,1,0,0,0,0,1,3
2,1,0,0,0,0,1,0,0,0,0,2
3,1,0,0,0,0,0,0,1,0,0,1
4,0,0,0,1,0,1,0,0,0,0,0


## 7. Data integration (merge internal sales with an external targets table)

In [13]:
att = W.integrate_targets(df)
print(att.groupby("region")["attainment_pct"].mean().round(1).sort_values())
att.head()

region
Central   95.60
West      95.70
East      95.90
South     97.40
North     99.40
Name: attainment_pct, dtype: float64


,region,year_month,revenue,monthly_target,attainment_pct
0,Central,2024-01,"1,943.66",6000,32.40
1,Central,2024-02,"3,208.56",6000,53.50
2,Central,2024-03,"2,985.91",6000,49.80
3,Central,2024-04,"5,264.15",6000,87.70
4,Central,2024-05,"6,570.71",6000,109.50


## 8. Wrangling log
| Step | Technique |
|---|---|
| SQL view -> DataFrame | `pd.read_sql` |
| Lifetime totals per customer | `groupby` + `merge(validate="m:1")` |
| Targets from outside source | `merge(how="left")` |
| Tidy vs wide | `pivot_table`, `melt`, `stack` |
| Business bands | `apply` / `lambda`, `qcut` |